# Simulated Data: ARI Analysis & Plots

Loads per-scenario ARI results produced by `01_simulated_kmeans.ipynb` and generates:
- **Violin plots** of ARI distributions across 30 runs per (scenario × method × target)
- **Summary table** with mean ± SD per group

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
# -------------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------------
SCENARIOS = ["balanced", "mild_imbalanced", "strong_imbalanced"]
OUTPUT_ROOT = NOTEBOOK_DIR / "outputs"

METHOD_ORDER = ["uncorrected", "central", "federated"]
METHOD_PALETTE = {
    "uncorrected": "#999999",
    "central":     "#fd8d3c",
    "federated":   "#2171b5",
}
SCENARIO_LABELS = {
    "balanced": "Balanced",
    "mild_imbalanced": "Mild Imbalanced",
    "strong_imbalanced": "Strong Imbalanced",
}

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

In [ ]:
# -------------------------------------------------------------------------
# Load all per-scenario result TSVs
# -------------------------------------------------------------------------

dfs = []
for scenario in SCENARIOS:
    path = OUTPUT_ROOT / f"{scenario}_ari_results.tsv"
    if not path.exists():
        print(f"WARNING: {path} not found — run 01_simulated_kmeans.ipynb first")
        continue
    df = pd.read_csv(path, sep="\t")
    dfs.append(df)
    print(f"Loaded {scenario}: {len(df)} rows")

results = pd.concat(dfs, ignore_index=True)
results["method"] = pd.Categorical(results["method"], categories=METHOD_ORDER, ordered=True)
results["scenario_label"] = results["scenario"].map(SCENARIO_LABELS)
print(f"\nTotal rows: {len(results)}")
display(results.head())

In [ ]:
# -------------------------------------------------------------------------
# Violin plots: one figure per target (condition / batch)
# x=scenario, hue=method, y=ARI
# -------------------------------------------------------------------------

for target in ["condition", "batch"]:
    subset = results[results["target"] == target]

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.violinplot(
        data=subset,
        x="scenario_label",
        y="ARI",
        hue="method",
        hue_order=METHOD_ORDER,
        palette=METHOD_PALETTE,
        order=[SCENARIO_LABELS[s] for s in SCENARIOS],
        inner="box",
        linewidth=0.8,
        ax=ax,
    )
    ax.axhline(0, color="gray", linewidth=0.7, linestyle="--", alpha=0.6)
    ax.set_title(f"ARI Distribution — target: {target}", fontsize=14)
    ax.set_xlabel("Scenario")
    ax.set_ylabel("ARI (30 runs)")
    ax.set_ylim(-0.3, 1.05)
    ax.legend(title="Method", loc="lower right", fontsize=9)
    fig.tight_layout()
    out_path = OUTPUT_ROOT / f"violin_ari_{target}.pdf"
    fig.savefig(out_path, bbox_inches="tight")
    print(f"Saved: {out_path}")
    plt.show()

In [ ]:
# -------------------------------------------------------------------------
# Summary table: mean ± SD per (scenario, method, target)
# -------------------------------------------------------------------------

summary = (
    results
    .groupby(["scenario", "method", "target"], observed=True)["ARI"]
    .agg(["mean", "std", "min", "max", "count"])
    .rename(columns={"mean": "ARI_mean", "std": "ARI_std", "min": "ARI_min",
                     "max": "ARI_max", "count": "n_runs"})
    .round(4)
    .reset_index()
)
summary["mean±SD"] = summary.apply(
    lambda r: f"{r['ARI_mean']:.3f} ± {r['ARI_std']:.3f}", axis=1
)

# Save
summary_path = OUTPUT_ROOT / "ari_summary.tsv"
summary.to_csv(summary_path, sep="\t", index=False)
print(f"Saved summary: {summary_path}")

# Display pivoted
pivot = summary.pivot_table(
    index=["scenario", "target"],
    columns="method",
    values="mean±SD",
    aggfunc="first",
)[METHOD_ORDER]
display(pivot)

In [ ]:
# -------------------------------------------------------------------------
# Heatmap: mean ARI per (scenario × method), faceted by target
# -------------------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, target in zip(axes, ["condition", "batch"]):
    heat_data = (
        summary[summary["target"] == target]
        .pivot(index="scenario", columns="method", values="ARI_mean")
        .reindex(index=SCENARIOS, columns=METHOD_ORDER)
    )
    sns.heatmap(
        heat_data,
        ax=ax,
        annot=True,
        fmt=".3f",
        cmap="RdYlGn",
        vmin=-0.1,
        vmax=1.0,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8},
    )
    ax.set_title(f"Mean ARI — {target}", fontsize=12)
    ax.set_ylabel("Scenario")
    ax.set_xlabel("")
    ax.set_yticklabels([SCENARIO_LABELS.get(s, s) for s in SCENARIOS], rotation=0)

fig.suptitle("Mean ARI across 30 Simulation Runs", fontsize=13, y=1.02)
fig.tight_layout()
heatmap_path = OUTPUT_ROOT / "heatmap_mean_ari.pdf"
fig.savefig(heatmap_path, bbox_inches="tight")
print(f"Saved: {heatmap_path}")
plt.show()